# Evaluación Etapa 2 - métricas de generación y fidelidad por atributo



In [ ]:
from __future__ import annotations

import math
import re
import sys
import unicodedata
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
    }
)

RNG = np.random.default_rng(20260914)

In [ ]:
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUNS_DIR = ROOT / "runs"
MANIFEST = ROOT / "data/manifests/m3di_test.parquet"
FIG_DIR = ROOT / "reports/figures/eval"
TAB_DIR = ROOT / "reports/tables/eval"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
FIELD_ALIASES = {
    "image_id": ["image_id", "id", "idx", "index"],
    "prediction": ["prediction", "pred", "output", "text", "caption_pred", "generated"],
    "reference": ["caption_ref", "caption", "reference", "caption_gt", "ref"],
}

# Modelos que recuperan de un pool en vez de generar: su BLEU no es comparable
# con el de los generativos sin proyección a conjunto cerrado (sección 9).
RETRIEVAL_MODELS = {"vljepa", "vl-jepa", "vl_jepa"}

N_PERMUTATION_PAIRS = 2000  # pares para estimar S_0
DISC_QUANTILE = 0.75  # umbral tau de la discordancia
N_BOOTSTRAP = 10_000


@dataclass
class AttrSpec:
    """Especificación de un atributo y de cómo extraerlo del texto."""

    name: str  # columna con el valor verdadero
    label: str | None = None  # nombre para tablas y figuras
    kind: str = "nominal"  # nominal | ordinal
    order: list | None = None  # orden de las clases (ordinal)
    surface: dict = field(default_factory=dict)  # valor GT -> formas superficiales
    anchors: tuple = ()  # el valor debe aparecer cerca de esto
    exclude_anchors: tuple = ()  # ...y lejos de esto
    window: int = 45  # ventana de contexto en caracteres
    pattern: str | None = None  # regex con grupo (?P<val>...) (override)

    def __post_init__(self):
        self.label = self.label or self.name


# Valores por defecto para Multimodal3DIdent. Ajustar tras leer el diagnóstico
# de la sección 5: esos son los captions reales, esto es solo una hipótesis.
POSITIONS_X = {0: ["left"], 1: ["center", "middle", "centre"], 2: ["right"]}
POSITIONS_Y = {0: ["top", "upper"], 1: ["center", "middle", "centre"], 2: ["bottom", "lower"]}

ATTRS: list[AttrSpec] = [
    AttrSpec(name="object_shape", label="forma"),
    AttrSpec(
        name="object_xpos",
        label="pos. x",
        kind="ordinal",
        order=[0, 1, 2],
        surface=POSITIONS_X,
        # "center" es compartido con el eje y: no se toma si le sigue izq/der.
        pattern=r"\b(?P<val>left|right|(?:centre|center|middle)(?![\s-]+(?:left|right)))\b",
    ),
    AttrSpec(
        name="object_ypos",
        label="pos. y",
        kind="ordinal",
        order=[0, 1, 2],
        surface=POSITIONS_Y,
        pattern=(
            r"\b(?P<val>top|upper|bottom|lower|centre|center|middle)\b"
            # ...o al final del texto ("visible at the center")
            r"(?=[\s-]+(?:left|right|centre|center|middle|of|part|side|region|corner|half)\b|\s*$)"
        ),
    ),
    AttrSpec(
        name="object_color",
        label="color objeto",
        # las anclas se completan con el vocabulario de formas (sección 5).
        anchors=("object", "shape", "colored", "coloured"),
        exclude_anchors=("background", "spotlight", "backdrop"),
        window=60,
    ),
    AttrSpec(
        name="spotlight_color",
        label="color foco",
        anchors=("spotlight", "lit", "illuminat"),
        exclude_anchors=("background", "backdrop"),
        window=60,
    ),
    AttrSpec(
        name="background_color",
        label="color fondo",
        anchors=("background", "backdrop", "behind"),
        exclude_anchors=("spotlight", "illuminat"),
        window=60,
    ),
]

In [ ]:
def resolve(cols: list[str], aliases: list[str]) -> str | None:
    lower = {c.lower(): c for c in cols}
    for a in aliases:
        if a.lower() in lower:
            return lower[a.lower()]
    return None


# ── Parseo de exp_id ─────────────────────────────────────────────────────────
# Formato: e{k}_{dataset}_{split}_{modelo}_p{j}[_{nombre-prompt}][_n{N}[_{sufijo}]]
#   p. ej. e1_m3di_test_internvl_p0_minimal_n1000
# El modelo se captura de forma no codiciosa hasta el primer _p{j}, por lo que
# admite guiones bajos (qwen2_5_vl); el nombre del prompt también.
EXP_COLS = ["exp", "dataset", "split", "model", "prompt_id", "prompt_name", "n_samples", "suffix"]
META_COLS = ["exp", "dataset", "split", "model", "prompt_id", "prompt_name", "prompt", "n_samples", "suffix"]
SPLITS = {"train", "val", "valid", "test"}
KEEP = {"dataset": "m3di", "split": "test"}  # filtra corridas por lo parseado (None = no filtrar)

EXP_RE = re.compile(
    r"^(?P<exp>e\d+)_(?P<dataset>[^_]+)_(?P<split>[^_]+)_"
    r"(?P<model>.+?)_(?P<prompt_id>p\d+)(?:_(?P<prompt_name>.+?))?"
    r"(?:_n(?P<n_samples>\d+)(?:_(?P<suffix>.+))?)?$",
    re.IGNORECASE,
)


def _parse_tokens(exp_id: str) -> dict | None:
    """Respaldo por tokens: ubica el split conocido y el primer p{j} posterior al modelo."""
    tok = exp_id.split("_")
    i_split = next((i for i, t in enumerate(tok) if t.lower() in SPLITS), None)
    i_p = next(
        (
            i
            for i, t in enumerate(tok)
            if re.fullmatch(r"p\d+", t, re.IGNORECASE) and (i_split is None or i > i_split + 1)
        ),
        None,
    )
    if i_split is None or i_p is None:
        return None
    rest = tok[i_p + 1 :]
    i_n = next((i for i, t in enumerate(rest) if re.fullmatch(r"n\d+", t)), None)
    return {
        "exp": "_".join(tok[: max(i_split - 1, 0)]) or None,
        "dataset": tok[i_split - 1] if i_split > 0 else None,
        "split": tok[i_split],
        "model": "_".join(tok[i_split + 1 : i_p]),
        "prompt_id": tok[i_p],
        "prompt_name": "_".join(rest[:i_n] if i_n is not None else rest) or None,
        "n_samples": rest[i_n][1:] if i_n is not None else None,
        "suffix": ("_".join(rest[i_n + 1 :]) or None) if i_n is not None else None,
    }


def parse_exp_id(exp_id: str) -> dict:
    """Devuelve {exp, dataset, split, model, prompt_id, prompt_name, prompt, n_samples, suffix, parse_ok}."""
    s = str(exp_id).strip()
    m = EXP_RE.match(s)
    d = m.groupdict() if m else _parse_tokens(s)
    if d is None:
        return {"exp_id": exp_id, "parse_ok": False, **dict.fromkeys(META_COLS)}
    d = {k: d.get(k) for k in EXP_COLS}
    for k in ("exp", "dataset", "split", "prompt_id"):
        d[k] = d[k].lower() if d[k] else d[k]
    d["n_samples"] = int(d["n_samples"]) if d["n_samples"] else None
    d["prompt"] = f"{d['prompt_id']}_{d['prompt_name']}" if d["prompt_name"] else d["prompt_id"]
    return {"exp_id": exp_id, "parse_ok": True, **d}


def load_runs(runs_dir: Path, manifest_path: Path | None = None) -> pl.DataFrame:
    files = sorted(p for p in runs_dir.glob("*.jsonl"))
    if not files:
        raise FileNotFoundError(f"No hay *.jsonl en {runs_dir}")

    manifest = None
    if manifest_path is not None and manifest_path.exists():
        manifest = pl.read_parquet(manifest_path)

    frames = []
    for f in files:
        exp_id = f.stem
        meta = parse_exp_id(exp_id)
        if not meta["parse_ok"]:
            print(f"  ! {exp_id}: exp_id sin formato reconocible, se omite")
            continue
        if KEEP and any(v is not None and meta[k] != v.lower() for k, v in KEEP.items()):
            print(f"  · {exp_id}: excluido por KEEP={KEEP}")
            continue

        df = pl.read_ndjson(f)
        cols = df.columns
        c_id = resolve(cols, FIELD_ALIASES["image_id"])
        c_pred = resolve(cols, FIELD_ALIASES["prediction"])
        c_ref = resolve(cols, FIELD_ALIASES["reference"])
        if c_pred is None:
            print(f"  ! {exp_id}: sin columna de predicción, se omite")
            continue
        if c_id is None:
            df = df.with_row_index("image_id")
            c_id = "image_id"

        keep = {c_id: "image_id", c_pred: "prediction"}
        if c_ref is not None:
            keep[c_ref] = "reference"
        present_attrs = [a.name for a in ATTRS if a.name in cols]
        keep.update({a: a for a in present_attrs})
        df = df.select(list(keep)).rename(keep)

        missing = [a.name for a in ATTRS if a.name not in df.columns]
        if missing and manifest is not None:
            mid = resolve(manifest.columns, FIELD_ALIASES["image_id"])
            have = [c for c in missing if c in manifest.columns]
            if mid and have:
                df = df.join(
                    manifest.select([mid] + have).rename({mid: "image_id"}),
                    on="image_id",
                    how="left",
                )
        if "reference" not in df.columns and manifest is not None:
            mid = resolve(manifest.columns, FIELD_ALIASES["image_id"])
            mref = resolve(manifest.columns, FIELD_ALIASES["reference"])
            if mid and mref:
                df = df.join(
                    manifest.select([mid, mref]).rename({mid: "image_id", mref: "reference"}),
                    on="image_id",
                    how="left",
                )

        df = df.with_columns(
            pl.lit(exp_id).alias("exp_id"),
            *[pl.lit(meta[k], dtype=pl.Utf8).alias(k) for k in META_COLS if k != "n_samples"],
            pl.lit(meta["n_samples"], dtype=pl.Int64).alias("n_samples"),
            pl.col("image_id").cast(pl.Utf8),
            pl.col("prediction").cast(pl.Utf8).fill_null(""),
        )
        frames.append(df)

    if not frames:
        raise ValueError(f"Ninguna corrida quedó tras el parseo/filtro. Archivos: {[f.stem for f in files][:5]}")
    preds = pl.concat(frames, how="diagonal_relaxed")
    print(f"Runs cargados: {preds['exp_id'].n_unique()}  |  filas: {preds.height:,}")
    return preds


preds = load_runs(RUNS_DIR, MANIFEST)
if "reference" not in preds.columns:
    preds = preds.with_columns(pl.lit(None, dtype=pl.Utf8).alias("reference"))
ATTRS = [a for a in ATTRS if a.name in preds.columns]
print("Atributos disponibles:", [a.name for a in ATTRS])

# Metadatos por corrida (una fila por exp_id) y diccionarios de acceso rápido
RUNS_META = (
    preds.select(["exp_id"] + META_COLS)
    .unique(subset=["exp_id"])
    .with_columns(pl.col("prompt_id").str.slice(1).cast(pl.Int64).alias("_p"))
    .sort(["model", "_p", "exp_id"])
    .drop("_p")
)
RUN_MODEL = dict(zip(RUNS_META["exp_id"], RUNS_META["model"]))
RUN_PROMPT = dict(zip(RUNS_META["exp_id"], RUNS_META["prompt"]))

dup = RUNS_META.group_by(["model", "prompt"]).agg(pl.col("exp_id")).filter(pl.col("exp_id").list.len() > 1)
if dup.height:
    print("⚠  Varias corridas comparten (modelo, prompt); distinguirlas por 'exp':")
    print(dup)

print(RUNS_META.select(["exp_id", "exp", "model", "prompt_id", "prompt_name", "prompt", "n_samples"]))
preds.head(3)

In [ ]:
# ── Discretización de atributos para el extractor ────────────────────────────
# Sin esto, cada tono continuo es una "clase" (≈10.000 valores ⇒ ≈10.000 regex por
# texto y atributo) y la forma se busca literalmente como "0".."6" en el texto.
import colorsys
import matplotlib.colors as mcolors

COLOR_ATTRS = ["object_color", "spotlight_color", "background_color"]
COLOR_CATS = ["red", "orange", "yellow", "green", "cyan", "blue", "purple", "pink"]
HUE_UPPER = [15, 40, 70, 160, 195, 255, 285, 345]  # límite superior (grados) de cada categoría


def hue_to_cat(h: float) -> str:
    deg = (float(h) * 360.0) % 360.0
    if deg >= 345 or deg < 15:
        return "red"
    return next(c for c, u in zip(COLOR_CATS[1:], HUE_UPPER[1:]) if deg < u)


def rgb_to_cat(rgb) -> str:
    h, s, v = colorsys.rgb_to_hsv(*rgb)
    return "gray" if (s < 0.2 or v < 0.2) else hue_to_cat(h)


# Colores con nombre de Matplotlib ("xkcd:bright yellow", "tab:green") -> categoría.
# Debe aplicarse ANTES de normalize(), que elimina los ':' y el nombre se vuelve irreconocible.
NAMED = {
    **{k.lower(): v for k, v in mcolors.XKCD_COLORS.items()},
    **{k.lower(): v for k, v in mcolors.TABLEAU_COLORS.items()},
    **{f"css:{k.lower()}": v for k, v in mcolors.CSS4_COLORS.items()},
}
PREFIX_RE = re.compile(r"\b(xkcd|tab|css4?)\s*:\s*", re.IGNORECASE)
NAME_WORD_RE = re.compile(r"[A-Za-z][A-Za-z'\-]*")


def replace_named_colors(text: str | None) -> str:
    if not text:
        return ""
    hits, pos = [], 0
    for m in PREFIX_RE.finditer(text):
        if m.start() < pos:
            continue
        pref = "css" if m.group(1).lower().startswith("css") else m.group(1).lower()
        words, j = [], m.end()
        for w in NAME_WORD_RE.finditer(text, m.end()):
            if w.start() != j and text[j : w.start()].strip():
                break
            words.append(w)
            j = w.end()
            if len(words) == 4:
                break
        for k in range(len(words), 0, -1):  # prefijo más largo que sea un nombre válido
            name = f"{pref}:{' '.join(w.group(0) for w in words[:k]).lower()}"
            if name in NAMED:
                hits.append((m.start(), words[k - 1].end(), rgb_to_cat(mcolors.to_rgb(NAMED[name]))))
                pos = words[k - 1].end()
                break
    for s, e, cat in reversed(hits):
        text = text[:s] + cat + text[e:]
    return text


SHAPE_LEXICON = {
    "teapot": ["teapot", "tea pot", "kettle"],
    "hare": ["hare", "rabbit", "bunny"],
    "dragon": ["dragon"],
    "cow": ["cow", "bull", "ox", "cattle", "calf"],
    "armadillo": ["armadillo"],
    "horse": ["horse", "pony", "stallion", "mare"],
    "head": ["head", "bust", "face"],
}
COLOR_SURFACE = {
    "red": ["red", "reddish", "crimson", "scarlet", "maroon", "burgundy", "ruby"],
    "orange": ["orange", "orangish", "amber", "tangerine", "peach", "rust", "copper", "salmon", "coral"],
    "yellow": ["yellow", "yellowish", "gold", "golden", "mustard", "lemon"],
    "green": ["green", "greenish", "lime", "olive", "chartreuse", "emerald", "mint"],
    "cyan": ["cyan", "teal", "turquoise", "aqua", "aquamarine"],
    "blue": ["blue", "bluish", "navy", "azure", "cobalt", "sapphire", "cerulean"],
    "purple": ["purple", "purplish", "violet", "lavender", "indigo", "lilac", "plum", "mauve"],
    "pink": ["pink", "pinkish", "magenta", "fuchsia", "rose"],
}
SHAPE_RX = {k: re.compile(r"\b(?:" + "|".join(map(re.escape, v)) + r")s?\b", re.I) for k, v in SHAPE_LEXICON.items()}


def first_shape(text: str | None) -> str | None:
    hits = [(m.start(), k) for k, rx in SHAPE_RX.items() for m in rx.finditer(text or "")]
    return min(hits)[1] if hits else None


# 1) Forma: código -> nombre, inferido de la referencia (moda por código) y su pureza
pairs = Counter(
    (c, first_shape(t)) for c, t in preds.unique(subset=["image_id"]).select(["object_shape", "reference"]).iter_rows()
)
SHAPE_NAMES = {}
for c in sorted({c for c, _ in pairs if c is not None}):
    cand = {s: n for (cc, s), n in pairs.items() if cc == c and s}
    SHAPE_NAMES[c] = max(cand, key=cand.get) if cand else str(c)
    tot = sum(n for (cc, _), n in pairs.items() if cc == c)
    print(f"  forma {c} -> {SHAPE_NAMES[c]:<10} pureza {cand.get(SHAPE_NAMES[c], 0) / tot:.3f}")
assert len(set(SHAPE_NAMES.values())) == len(SHAPE_NAMES), f"Asignación no biyectiva: {SHAPE_NAMES}"

# 2) Reemplaza códigos y tonos por etiquetas discretas (mismos nombres de columna: ATTRS no cambia)
preds = preds.with_columns(
    pl.col("object_shape").map_elements(lambda c: SHAPE_NAMES.get(c, str(c)), return_dtype=pl.Utf8),
    *[pl.col(c).map_elements(hue_to_cat, return_dtype=pl.Utf8) for c in COLOR_ATTRS if c in preds.columns],
)

# 3) Formas superficiales para el extractor y sinónimos de forma como anclas del color del objeto
for spec in ATTRS:
    if spec.name == "object_shape":
        spec.surface = SHAPE_LEXICON
    elif spec.name in COLOR_ATTRS:
        spec.surface = COLOR_SURFACE
    if spec.name == "object_color":
        spec.anchors = tuple(dict.fromkeys(spec.anchors + tuple(w for v in SHAPE_LEXICON.values() for w in v)))
print("Atributos discretizados:", {c: preds[c].n_unique() for c in ["object_shape"] + COLOR_ATTRS if c in preds.columns})


In [ ]:
PREAMBLE_RE = re.compile(
    r"^\s*(?:sure[,!.]?\s*)?(?:the\s+)?(?:image|picture|photo|scene)\s+"
    r"(?:shows|depicts|displays|features|contains|is\s+of)\s*:?\s*",
    flags=re.IGNORECASE,
)
PUNCT_RE = re.compile(r"[^\w\s]", flags=re.UNICODE)
WS_RE = re.compile(r"\s+")


def normalize(text: str | None) -> str:
    if not text:
        return ""
    t = unicodedata.normalize("NFKC", str(text)).strip()
    t = t.split("\n")[0] if t.count("\n") and len(t.split("\n")[0]) > 15 else t.replace("\n", " ")
    t = PREAMBLE_RE.sub("", t)
    t = t.lower()
    t = PUNCT_RE.sub(" ", t)
    return WS_RE.sub(" ", t).strip()


def tokenize(text: str) -> list[str]:
    return normalize(text).split()


preds = preds.with_columns(
    pl.col("prediction").map_elements(lambda t: normalize(replace_named_colors(t)), return_dtype=pl.Utf8).alias("pred_norm"),
    pl.col("reference").map_elements(lambda t: normalize(replace_named_colors(t)), return_dtype=pl.Utf8).alias("ref_norm"),
)

In [ ]:
def ngrams(toks: list[str], n: int) -> Counter:
    return Counter(tuple(toks[i : i + n]) for i in range(len(toks) - n + 1))


def sentence_bleu(cand: list[str], ref: list[str], max_n: int = 4) -> float:
    """BLEU-n de oración con suavizado Chen & Cherry (2014), método 3."""
    if not cand or not ref:
        return 0.0
    logs, invcnt = [], 1
    for n in range(1, max_n + 1):
        cc, rc = ngrams(cand, n), ngrams(ref, n)
        denom = sum(cc.values())
        if denom == 0:
            return 0.0  # candidato más corto que n
        match = sum(min(c, rc[g]) for g, c in cc.items())
        if match == 0:
            invcnt *= 2
            p = 1.0 / (invcnt * denom)
        else:
            p = match / denom
        logs.append(math.log(p))
    c, r = len(cand), len(ref)
    bp = 1.0 if c > r else math.exp(1 - r / c)
    return bp * math.exp(sum(logs) / max_n)


def lcs_len(a: list[str], b: list[str]) -> int:
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(cur[j], prev[j + 1]))
        prev = cur
    return prev[-1]


def rouge_l(cand: list[str], ref: list[str], beta: float = 1.2) -> float:
    if not cand or not ref:
        return 0.0
    l = lcs_len(cand, ref)
    if l == 0:
        return 0.0
    p, r = l / len(cand), l / len(ref)
    return ((1 + beta**2) * p * r) / (r + beta**2 * p)


class CiderD:
    """CIDEr-D con una referencia por imagen. IDF estimado sobre el corpus de
    referencias: con captions plantillados el IDF se degenera y el puntaje
    discrimina poco — es un resultado esperado, no un error."""

    def __init__(self, ref_token_lists: list[list[str]], n: int = 4, sigma: float = 6.0):
        self.n, self.sigma = n, sigma
        self.df = [Counter() for _ in range(n)]
        for toks in ref_token_lists:
            for k in range(n):
                for g in set(ngrams(toks, k + 1)):
                    self.df[k][g] += 1
        self.log_N = math.log(max(len(ref_token_lists), 1))

    def _vec(self, toks: list[str]):
        vecs, norms = [], []
        for k in range(self.n):
            v = {}
            for g, cnt in ngrams(toks, k + 1).items():
                idf = self.log_N - math.log(max(self.df[k].get(g, 0), 1))
                v[g] = cnt * idf
            vecs.append(v)
            norms.append(math.sqrt(sum(x * x for x in v.values())))
        return vecs, norms

    def score(self, cand: list[str], ref: list[str]) -> float:
        vc, nc = self._vec(cand)
        vr, nr = self._vec(ref)
        delta = len(cand) - len(ref)
        total = 0.0
        for k in range(self.n):
            if nc[k] == 0 or nr[k] == 0:
                continue
            s = sum(min(w, vr[k].get(g, 0.0)) * vr[k].get(g, 0.0) for g, w in vc[k].items())
            total += (s / (nc[k] * nr[k])) * math.exp(-(delta**2) / (2 * self.sigma**2))
        return 10.0 * total / self.n

In [ ]:
ref_lookup = preds.select(["image_id", "ref_norm"]).unique(subset=["image_id"]).drop_nulls()
ref_tokens_corpus = [t.split() for t in ref_lookup["ref_norm"].to_list() if t]
cider = CiderD(ref_tokens_corpus)
print(f"Corpus de referencias para IDF: {len(ref_tokens_corpus):,} captions")

rows = []
for r in preds.select(["exp_id", "image_id", "pred_norm", "ref_norm"]).iter_rows(named=True):
    c, g = r["pred_norm"].split(), (r["ref_norm"] or "").split()
    rows.append(
        {
            "exp_id": r["exp_id"],
            "image_id": r["image_id"],
            "bleu4": sentence_bleu(c, g),
            "rouge_l": rouge_l(c, g),
            "cider_d": cider.score(c, g),
            "n_tokens": len(c),
            "empty": len(c) == 0,
        }
    )
text_long = pl.DataFrame(rows)

text_agg = (
    text_long.group_by("exp_id")
    .agg(
        pl.col("bleu4").mean(),
        pl.col("rouge_l").mean(),
        pl.col("cider_d").mean(),
        pl.col("n_tokens").mean().alias("tokens_medios"),
        pl.col("empty").mean().alias("tasa_vacias"),
    )
    .sort("exp_id")
)
text_agg

In [ ]:
def surface_forms(spec: AttrSpec, values: list) -> dict:
    out = {}
    for v in values:
        if v is None:
            continue
        forms = spec.surface.get(v)
        if forms is None:
            forms = [str(v).replace("_", " ").replace("-", " ").lower().strip()]
        out[v] = sorted({normalize(f) for f in forms if normalize(f)}, key=len, reverse=True)
    return out


def build_matcher(spec: AttrSpec, values: list):
    forms = surface_forms(spec, values)
    pairs = []  # (regex, valor)
    for v, fs in forms.items():
        for f in fs:
            pairs.append((re.compile(rf"(?<!\w){re.escape(f)}(?!\w)"), v))
    anchors = [re.compile(re.escape(a.lower())) for a in spec.anchors]
    excl = [re.compile(re.escape(a.lower())) for a in spec.exclude_anchors]
    override = re.compile(spec.pattern, re.IGNORECASE) if spec.pattern else None

    def extract(text: str):
        t = normalize(text)
        if not t:
            return None
        if override is not None:
            m = override.search(t)
            if not m:
                return None
            cand = normalize(m.group("val"))
            for v, fs in forms.items():
                if cand in fs:
                    return v
            return None

        hits = [(m.start(), m.end(), v) for rx, v in pairs for m in rx.finditer(t)]
        if not hits:
            return None
        if not anchors:
            return min(hits, key=lambda h: h[0])[2]

        # Asignación competitiva: cada ocurrencia se queda con el atributo cuya
        # ancla tenga más cerca. Excluir por mera presencia en la ventana falla
        # cuando los tres colores conviven en la misma oración.
        best, best_d = None, math.inf
        for s_, e_, v in hits:
            lo, hi = max(0, s_ - spec.window), min(len(t), e_ + spec.window)
            ctx, off = t[lo:hi], lo
            d_own = min(
                (abs(m.start() + off - s_) for rx in anchors for m in rx.finditer(ctx)),
                default=math.inf,
            )
            d_excl = min(
                (abs(m.start() + off - s_) for rx in excl for m in rx.finditer(ctx)),
                default=math.inf,
            )
            if d_own is math.inf or d_excl < d_own:
                continue
            if d_own < best_d:
                best, best_d = v, d_own
        return best

    return extract


# El color del objeto aparece junto al sustantivo de la forma: se usan las
# propias formas como anclas léxicas.
shape_specs = [a for a in ATTRS if a.name == "object_shape"]
if shape_specs:
    shape_words = tuple(
        normalize(str(v)) for v in preds["object_shape"].unique().to_list() if v is not None
    )
    for spec in ATTRS:
        if spec.name == "object_color":
            spec.anchors = tuple(dict.fromkeys(spec.anchors + shape_words))

EXTRACTORS = {}
for spec in ATTRS:
    vals = [v for v in preds[spec.name].unique().to_list() if v is not None]
    EXTRACTORS[spec.name] = build_matcher(spec, sorted(vals, key=str))
    print(f"{spec.name:<18} |A_k| = {len(vals):>2}  valores: {sorted(map(str, vals))[:8]}")

In [ ]:
ref_gt = preds.unique(subset=["image_id"]).select(
    ["image_id", "ref_norm"] + [a.name for a in ATTRS]
)

val_rows = []
failures = {}
for spec in ATTRS:
    f = EXTRACTORS[spec.name]
    ok, tot, bad = 0, 0, []
    for r in ref_gt.iter_rows(named=True):
        if r[spec.name] is None or not r["ref_norm"]:
            continue
        tot += 1
        got = f(r["ref_norm"])
        if got == r[spec.name]:
            ok += 1
        elif len(bad) < 5:
            bad.append((r["ref_norm"], r[spec.name], got))
    val_rows.append({"atributo": spec.label, "n": tot, "exactitud_extractor": ok / max(tot, 1)})
    failures[spec.label] = bad

extractor_report = pl.DataFrame(val_rows)
print(extractor_report)

UMBRAL = 0.99
malos = extractor_report.filter(pl.col("exactitud_extractor") < UMBRAL)
if malos.height:
    print(f"\n⚠  Atributos bajo {UMBRAL:.0%} — ajustar ATTRS antes de interpretar nada:\n")
    for lab in malos["atributo"].to_list():
        print(f"--- {lab} ---")
        for txt, esperado, obtenido in failures[lab]:
            print(f"   esperado={esperado!r:>12}  obtenido={obtenido!r:>12}  | {txt[:95]}")
        print()
else:
    print(f"\n✓ Extractor válido (≥ {UMBRAL:.0%} en todos los atributos).")

In [ ]:
attr_rows = []
for spec in ATTRS:
    f = EXTRACTORS[spec.name]
    for r in preds.select(
        ["exp_id", "model", "prompt", "image_id", "pred_norm", spec.name]
    ).iter_rows(named=True):
        y_true = r[spec.name]
        if y_true is None:
            continue
        y_pred = f(r["pred_norm"])
        attr_rows.append(
            {
                "exp_id": r["exp_id"],
                "model": r["model"],
                "prompt": r["prompt"],
                "image_id": r["image_id"],
                "attribute": spec.label,
                "y_true": str(y_true),
                "y_pred": None if y_pred is None else str(y_pred),
                "covered": y_pred is not None,
                "correct": y_pred == y_true,
            }
        )

attr_long = pl.DataFrame(attr_rows, infer_schema_length=None)
attr_long.write_parquet(TAB_DIR / "attribute_long.parquet")
print(f"Tabla larga: {attr_long.height:,} filas -> {TAB_DIR / 'attribute_long.parquet'}")


def chance_agreement(df: pl.DataFrame) -> float:
    """p_0 = sum_v q(v) * q_hat(v), acuerdo esperado bajo marginales independientes."""
    n = df.height
    if n == 0:
        return 0.0
    q = df.group_by("y_true").len().with_columns(pl.col("len") / n)
    qh = df.group_by("y_pred").len().with_columns(pl.col("len") / n)
    j = q.join(qh, left_on="y_true", right_on="y_pred", how="inner")
    return float((j["len"] * j["len_right"]).sum()) if j.height else 0.0


agg = []
for (exp_id, attribute), g in attr_long.group_by(["exp_id", "attribute"], maintain_order=True):
    p0 = chance_agreement(g)
    acc = float(g["correct"].mean())
    cov = float(g["covered"].mean())
    cond = (
        float(g.filter(pl.col("covered"))["correct"].mean()) if g["covered"].any() else float("nan")
    )
    agg.append(
        {
            "exp_id": exp_id,
            "attribute": attribute,
            "cobertura": cov,
            "acc_estricta": acc,
            "acc_condicional": cond,
            "kappa": (acc - p0) / (1 - p0) if p0 < 1 else float("nan"),
        }
    )
attr_agg = pl.DataFrame(agg).sort(["exp_id", "attribute"])

exact_match = (
    attr_long.group_by(["exp_id", "image_id"])
    .agg(pl.col("correct").all().alias("em"), pl.col("correct").mean().alias("acc_img"))
    .group_by("exp_id")
    .agg(pl.col("em").mean().alias("exact_match"), pl.col("acc_img").mean().alias("acc_macro"))
)

summary = (
    text_agg.join(exact_match, on="exp_id", how="left")
    .join(
        attr_agg.group_by("exp_id").agg(pl.col("kappa").mean().alias("kappa_macro")),
        on="exp_id",
        how="left",
    )
    .join(RUNS_META.select(["exp_id", "exp", "model", "prompt_id", "prompt"]), on="exp_id", how="left")
    .sort(["model", "prompt"])
)
summary.write_csv(TAB_DIR / "summary_by_run.csv")
attr_agg.write_csv(TAB_DIR / "attribute_metrics.csv")
summary.select(
    [
        "model",
        "prompt",
        "bleu4",
        "bleu4_norm",
        "rouge_l",
        "cider_d",
        "acc_macro",
        "exact_match",
        "kappa_macro",
    ]
)

In [ ]:
print([a.name for a in ATTRS])
print("object_color_name" in preds.columns)

In [ ]:
attr_agg.pivot(values="acc_estricta", index="exp_id", on="attribute")

In [ ]:
# El cuartil se toma por rango y no por valor: con captions plantillados hay
# muchos BLEU empatados (varios exactamente 1.0) y un umbral por valor con ">"
# puede dejar el subconjunto vacío.
top = (
    text_long.with_columns(
        pl.col("bleu4").rank("ordinal", descending=True).over("exp_id").alias("rk"),
        pl.len().over("exp_id").alias("n_run"),
    )
    .filter(pl.col("rk") <= ((1 - DISC_QUANTILE) * pl.col("n_run")).ceil())
    .select(["exp_id", "image_id", "bleu4"])
)
disc = (
    attr_long.join(top, on=["exp_id", "image_id"])
    .group_by(["exp_id", "attribute"])
    .agg(
        (1 - pl.col("correct").mean()).alias("disc"),
        pl.len().alias("n"),
    )
    .sort(["exp_id", "attribute"])
)
disc.write_csv(TAB_DIR / "discordancia.csv")
disc.pivot(values="disc", index="exp_id", on="attribute")

In [ ]:
def paired_bootstrap(a: np.ndarray, b: np.ndarray, B: int = N_BOOTSTRAP):
    d = a - b
    idx = RNG.integers(0, len(d), size=(B, len(d)))
    boot = d[idx].mean(axis=1)
    lo, hi = np.quantile(boot, [0.025, 0.975])
    p = 2 * min((boot <= 0).mean(), (boot >= 0).mean())
    return float(d.mean()), float(lo), float(hi), float(min(p, 1.0))


def mcnemar(a: np.ndarray, b: np.ndarray):
    b_cnt = int(np.sum(a & ~b))
    c_cnt = int(np.sum(~a & b))
    n = b_cnt + c_cnt
    if n == 0:
        return b_cnt, c_cnt, 1.0
    if n <= 1000:
        k = min(b_cnt, c_cnt)
        p = 2 * sum(math.comb(n, i) for i in range(k + 1)) / (2**n)
    else:
        z = (abs(b_cnt - c_cnt) - 1) / math.sqrt(n)
        p = math.erfc(z / math.sqrt(2))
    return b_cnt, c_cnt, float(min(p, 1.0))


def holm(pvals: list[float]) -> list[float]:
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * pvals[i])
        adj[i] = min(running, 1.0)
    return adj.tolist()


best_run = (
    summary.sort("acc_macro", descending=True)
    .group_by("model", maintain_order=True)
    .first()
    .select(["model", "exp_id", "acc_macro"])
)
print("Mejor prompt por modelo:")
print(best_run)

per_image = attr_long.group_by(["exp_id", "image_id"]).agg(
    pl.col("correct").mean().alias("acc_img"), pl.col("correct").all().alias("em")
)

comps, pv = [], []
models = best_run["model"].to_list()
exp_of = dict(zip(best_run["model"], best_run["exp_id"]))
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        m1, m2 = models[i], models[j]
        d1 = per_image.filter(pl.col("exp_id") == exp_of[m1]).sort("image_id")
        d2 = per_image.filter(pl.col("exp_id") == exp_of[m2]).sort("image_id")
        common = set(d1["image_id"]) & set(d2["image_id"])
        d1 = d1.filter(pl.col("image_id").is_in(common)).sort("image_id")
        d2 = d2.filter(pl.col("image_id").is_in(common)).sort("image_id")
        delta, lo, hi, p_boot = paired_bootstrap(d1["acc_img"].to_numpy(), d2["acc_img"].to_numpy())
        b_c, c_c, p_mc = mcnemar(d1["em"].to_numpy(), d2["em"].to_numpy())
        comps.append(
            {
                "modelo_A": m1,
                "modelo_B": m2,
                "n": len(common),
                "delta_acc": delta,
                "ic_95": f"[{lo:+.4f}, {hi:+.4f}]",
                "p_bootstrap": p_boot,
                "b": b_c,
                "c": c_c,
                "p_mcnemar_EM": p_mc,
            }
        )
        pv.append(p_mc)

if comps:
    comparisons = pl.DataFrame(comps).with_columns(pl.Series("p_mcnemar_holm", holm(pv)))
    comparisons.write_csv(TAB_DIR / "comparaciones_pareadas.csv")
    print(comparisons)
else:
    comparisons = pl.DataFrame()
    print("Se necesita más de un modelo para comparar.")

In [ ]:
def load_pool(runs_dir: Path) -> list[str] | None:
    pools = sorted(runs_dir.glob("*.pool.parquet"))
    if not pools:
        return None
    df = pl.read_parquet(pools[0])
    col = resolve(df.columns, FIELD_ALIASES["reference"] + ["pool_caption", "text"])
    return [normalize(t) for t in df[col].to_list()] if col else None


def tfidf_matrix(docs: list[str], vocab: dict | None = None):
    if vocab is None:
        vocab = {w: i for i, w in enumerate(sorted({w for d in docs for w in d.split()}))}
    X = np.zeros((len(docs), len(vocab)), dtype=np.float32)
    for r, d in enumerate(docs):
        for w, c in Counter(d.split()).items():
            if w in vocab:
                X[r, vocab[w]] = c
    return X, vocab


pool = load_pool(RUNS_DIR)
closed_summary = pl.DataFrame()
if pool is None:
    print("No se encontró *.pool.parquet — sección omitida.")
else:
    P, vocab = tfidf_matrix(pool)
    df_doc = (P > 0).sum(axis=0)
    idf = np.log(len(pool) / np.maximum(df_doc, 1)).astype(np.float32)
    P = P * idf
    P /= np.maximum(np.linalg.norm(P, axis=1, keepdims=True), 1e-9)

    closed_rows = []
    for exp_id in preds["exp_id"].unique().to_list():
        sub = preds.filter(pl.col("exp_id") == exp_id)
        C, _ = tfidf_matrix(sub["pred_norm"].to_list(), vocab)
        C = C * idf
        C /= np.maximum(np.linalg.norm(C, axis=1, keepdims=True), 1e-9)
        nearest = np.asarray(C @ P.T).argmax(axis=1)
        projected = [pool[k] for k in nearest]
        for spec in ATTRS:
            f = EXTRACTORS[spec.name]
            truth = sub[spec.name].to_list()
            ok = [f(p) == t for p, t in zip(projected, truth) if t is not None]
            closed_rows.append(
                {
                    "exp_id": exp_id,
                    "attribute": spec.label,
                    "acc_cerrada": float(np.mean(ok)) if ok else float("nan"),
                }
            )
    closed_summary = pl.DataFrame(closed_rows).pivot(
        values="acc_cerrada", index="exp_id", on="attribute"
    )
    closed_summary.write_csv(TAB_DIR / "acc_conjunto_cerrado.csv")
    print(closed_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.4))
mk = ["o", "s", "^", "D", "v", "P", "X", "*"]
prompts = sorted(summary["prompt"].unique().to_list())
for mi, model in enumerate(sorted(summary["model"].unique().to_list())):
    s = summary.filter(pl.col("model") == model)
    for r in s.iter_rows(named=True):
        ax.scatter(
            r["bleu4_norm"],
            r["acc_macro"],
            marker=mk[prompts.index(r["prompt"]) % len(mk)],
            s=70,
            color=f"C{mi}",
            edgecolor="white",
            linewidth=0.6,
            label=model if r["prompt"] == s["prompt"][0] else None,
        )
ax.axvline(0, color="0.4", lw=0.8, ls="--")
ax.text(
    0.005,
    0.02,
    "$\\tilde S=0$: nivel del azar",
    transform=ax.get_xaxis_transform(),
    fontsize=7,
    color="0.35",
    rotation=90,
    va="bottom",
)
ax.set_xlabel("BLEU-4 normalizado  $\\tilde S$")
ax.set_ylabel("Exactitud macro por atributo")
ax.set_title("Similitud textual vs. fidelidad de atributos\n(un punto por modelo × prompt)")
ax.legend(frameon=False, fontsize=8, title="modelo", title_fontsize=8)
fig.savefig(FIG_DIR / "plano_bleu_vs_atributos.png")
plt.show()

In [ ]:
best_ids = best_run["exp_id"].to_list()
sub = attr_agg.filter(pl.col("exp_id").is_in(best_ids))
attrs_order = [a.label for a in ATTRS if a.label in sub["attribute"].unique().to_list()]
models_order = sorted({RUN_MODEL[e] for e in best_ids})

fig, ax = plt.subplots(figsize=(7.2, 4.0))
w = 0.8 / max(len(models_order), 1)
x = np.arange(len(attrs_order))
for mi, model in enumerate(models_order):
    eid = exp_of[model]
    vals = [
        float(sub.filter((pl.col("exp_id") == eid) & (pl.col("attribute") == a))["acc_estricta"][0])
        if sub.filter((pl.col("exp_id") == eid) & (pl.col("attribute") == a)).height
        else np.nan
        for a in attrs_order
    ]
    ax.bar(x + mi * w - 0.4 + w / 2, vals, width=w * 0.92, label=model, color=f"C{mi}")
ax.set_xticks(x, attrs_order, rotation=15)
ax.set_ylabel("Exactitud estricta")
ax.set_ylim(0, 1)
ax.set_title("Exactitud por atributo — mejor prompt de cada modelo")
ax.legend(frameon=False, fontsize=8, ncols=len(models_order))
fig.savefig(FIG_DIR / "exactitud_por_atributo.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.2))
for mi, model in enumerate(models_order):
    s = attr_agg.filter(pl.col("exp_id") == exp_of[model])
    ax.scatter(
        s["cobertura"],
        s["acc_condicional"],
        s=60,
        color=f"C{mi}",
        label=model,
        edgecolor="white",
        linewidth=0.6,
    )
    for r in s.iter_rows(named=True):
        ax.annotate(
            r["attribute"],
            (r["cobertura"], r["acc_condicional"]),
            fontsize=6.5,
            xytext=(3, 3),
            textcoords="offset points",
            color="0.3",
        )
ax.set_xlabel("Cobertura  $\\mathrm{Cob}_k$")
ax.set_ylabel("Exactitud condicional  $\\mathrm{Acc}_k^{\\mathrm{cond}}$")
ax.set_title("Omitir no es equivocarse")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG_DIR / "cobertura_vs_condicional.png")
plt.show()

In [ ]:
worst = (
    attr_agg.filter(pl.col("exp_id").is_in(best_ids))
    .group_by("attribute")
    .agg(pl.col("acc_estricta").mean())
    .sort("acc_estricta")
)
worst_attr = worst["attribute"][0]
print(f"Atributo peor resuelto: {worst_attr}")

fig, axes = plt.subplots(
    1, len(models_order), figsize=(3.1 * len(models_order), 3.1), squeeze=False
)
for mi, model in enumerate(models_order):
    g = attr_long.filter((pl.col("exp_id") == exp_of[model]) & (pl.col("attribute") == worst_attr))
    labels = sorted({*g["y_true"].to_list()})
    cols = labels + ["⊥"]
    M = np.zeros((len(labels), len(cols)))
    for r in g.iter_rows(named=True):
        M[labels.index(r["y_true"]), cols.index(r["y_pred"] or "⊥")] += 1
    M = M / np.maximum(M.sum(axis=1, keepdims=True), 1)
    a = axes[0, mi]
    a.imshow(M, vmin=0, vmax=1, cmap="Blues")
    a.set_xticks(range(len(cols)), cols, rotation=90, fontsize=6)
    a.set_yticks(range(len(labels)), labels, fontsize=6)
    a.set_title(model, fontsize=9)
    a.grid(False)
    if mi == 0:
        a.set_ylabel("verdadero")
fig.suptitle(f"Confusión — {worst_attr} (filas normalizadas)", fontsize=10)
fig.savefig(FIG_DIR / f"confusion_{worst_attr.replace(' ', '_').replace('.', '')}.png")
plt.show()

In [ ]:
print("Tablas:")
for p in sorted(TAB_DIR.glob("*")):
    print("  ", p.relative_to(ROOT))
print("Figuras:")
for p in sorted(FIG_DIR.glob("*.png")):
    print("  ", p.relative_to(ROOT))

print("\nPara el informe, reportar explícitamente:")
print(
    f"  · Línea base por permutación S_0: BLEU={S0['bleu4']:.4f}, "
    f"ROUGE-L={S0['rouge_l']:.4f}, CIDEr-D={S0['cider_d']:.4f}"
)
print("  · Exactitud del extractor sobre referencias (tabla de la sección 5.1)")
print("  · Regla de normalización de texto (sección 3) y criterio de selección de prompt")
print("  · CIDEr-D es poco discriminativo con captions plantillados: su IDF se degenera")